# Convergence of Algorithms

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caam37830-sci-comp/notes/blob/main/01_analysis/convergence.ipynb)

Many numerical algorithms are *iterative*: instead of computing the answer directly, they produce a sequence of approximations $x_0, x_1, x_2, \dots$ that get closer and closer to the true solution.  Examples include Newton's method for finding roots, gradient descent for optimization, and iterative solvers for linear systems.

We'll let $x_\ast$ denote the true solution, and $x_k$ denote the $k$th iterate of an algorithm.  The *error* at iteration $k$ is

$$
\epsilon_k = |x_k - x_\ast|
$$

(if the $x_k$ are vectors, we use a norm instead: $\epsilon_k = \|x_k - x_\ast\|$).  An algorithm *converges* if

$$
\lim_{k\to\infty} \epsilon_k = 0
$$

Knowing that an algorithm converges is not enough to make it useful.  An algorithm that needs $5$ iterations to get $15$ correct digits is very different from one that needs $10^{30}$ iterations.  In this section, we'll make "how fast" precise, look at concrete algorithms that converge at different speeds, and see how to measure the speed from experimental data.

## Monitoring Convergence in Practice

In practice, we typically don't know the value of $x_\ast$ (otherwise we wouldn't need the algorithm).  There are two common substitutes for the error:

1. Run the algorithm for $N$ iterations and treat the last iterate as the true solution, i.e. estimate $\tilde{\epsilon}_k = |x_k - x_N|$.  This is reasonable for $k$ much smaller than $N$.
2. Monitor the difference between consecutive iterates, $\delta_k = |x_k - x_{k-1}|$.  If the algorithm is converging, then $\delta_k \to 0$ as well.  A typical *stopping criterion* is to stop as soon as $\delta_k < \mathrm{tol}$ (absolute tolerance) or $\delta_k < \mathrm{tol}\cdot|x_k|$ (relative tolerance).

In this class, we aren't going to worry too much about proving that algorithms converge.  However, we do want to be able to verify that an algorithm is converging, measure the *rate* of convergence, and compare two algorithms using experimental convergence data.

## Rate of Convergence

There are a [variety of ways](https://en.wikipedia.org/wiki/Rate_of_convergence) to define the rate of convergence.  The most common one compares the error at iteration $k+1$ to the error at iteration $k$.  We say that $\epsilon_k \to 0$ with **order** $q \ge 1$ and **rate** $\mu$ if

$$
\lim_{k\to \infty} \frac{\epsilon_{k+1}}{\epsilon_k^q} = \mu, \qquad 0 < \mu < \infty
$$

In words: once $k$ is large enough, each iteration does roughly $\epsilon_{k+1} \approx \mu\, \epsilon_k^q$.  The constant $\mu$ is also called the *asymptotic error constant*.

The most important cases have names:

| Name | Definition | What happens to the error |
|---|---|---|
| **linear** | $q = 1$ and $0 < \mu < 1$ | multiplied by $\mu$ each iteration, so $\epsilon_k \approx \mu^k \epsilon_0$ |
| **superlinear** | $\epsilon_{k+1}/\epsilon_k \to 0$ | faster than linear with any rate $\mu$; this includes every order $q > 1$ |
| **quadratic** | $q = 2$ | $\epsilon_{k+1} \approx \mu\, \epsilon_k^2$ |
| **cubic** | $q = 3$ | $\epsilon_{k+1} \approx \mu\, \epsilon_k^3$ |
| **sublinear** | $q = 1$ and $\mu = 1$ | slower than linear, e.g. $\epsilon_k = 1/k$ |

Larger values of $q$ indicate faster convergence.  For the same $q$, a smaller $\mu$ is faster.

A few remarks:
* If $q = 1$ and $\mu > 1$, the error grows.  An order $q < 1$ is also impossible for a sequence that converges to $0$.
* For $q > 1$, any $\mu > 0$ is allowed, even $\mu > 1$, because $\epsilon_{k+1}/\epsilon_k \approx \mu\, \epsilon_k^{q-1} \to 0$.
* The order and rate are *asymptotic* properties.  They say nothing about the first few iterations, and it can take many iterations before an algorithm settles into its asymptotic behavior.
* For some algorithms, the ratio $\epsilon_{k+1}/\epsilon_k$ has no limit, so they don't fit the definition exactly.  If the error is *bounded* by a sequence that converges linearly, e.g. $\epsilon_k \le C\, 2^{-k}$, we still say the algorithm converges linearly.  Bisection is an example (see below).

### Counting Correct Digits

The easiest way to understand the order $q$ is to count digits.  The quantity $d_k = -\log_{10} \epsilon_k$ is roughly the number of correct decimal digits in $x_k$: for example, $\epsilon_k = 10^{-6}$ means about $6$ correct digits.  Taking $-\log_{10}$ of $\epsilon_{k+1} \approx \mu\, \epsilon_k^q$ gives

$$
d_{k+1} \approx q\, d_k + \log_{10}(1/\mu)
$$

* **Linear** ($q = 1$): $d_{k+1} \approx d_k + \log_{10}(1/\mu)$.  Each iteration adds a *fixed number* of correct digits.  With $\mu = 1/2$ we gain $\log_{10} 2 \approx 0.3$ digits per iteration, so $16$ digits (double precision) take about $16/0.3 \approx 53$ iterations.  With $\mu = 0.1$ we gain exactly one digit per iteration.
* **Quadratic** ($q = 2$): $d_{k+1} \approx 2 d_k$.  The number of correct digits roughly *doubles* each iteration: $1, 2, 4, 8, 16$.  Starting from one correct digit, we reach double precision in about $4$ iterations.
* **Order $q$**: the number of correct digits is multiplied by about $q$ each iteration.
* **Sublinear**: the number of digits gained per iteration goes to $0$.  For $\epsilon_k = 1/k$, getting $16$ digits takes $10^{16}$ iterations.

### Model Sequences

Let's generate some model error sequences by running the recursion $\epsilon_{k+1} = \mu\, \epsilon_k^q$ directly. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# generates a linearly converging sequence: eps[k+1] = 0.5 * eps[k]
n = 50
mu = 0.5
q = 1
eps1 = [1.0]
for i in range(n):
    eps1.append(mu * eps1[-1]**q)

# generates a quadratically converging sequence: eps[k+1] = 2 * eps[k]**2
n = 9
mu = 2.0
q = 2
eps2 = [0.2]
for i in range(n):
    eps2.append(mu * eps2[-1]**q)

# generates a sublinearly converging sequence: eps[k] = 1/k
n = 50
eps3 = []
for k in range(1, n):
    eps3.append(1.0/k)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].semilogy(eps1, 'o-', label='linear')
ax[0].semilogy(eps2, 's-', label='quadratic')
ax[0].semilogy(eps3, '^-', label='sublinear')
ax[0].set_xlabel('k')
ax[0].set_ylabel(r"$\epsilon_k$")
ax[0].set_ylim(1e-16, 1e0)
ax[0].set_title("types of convergence (semilogy)")
ax[0].legend()

k = np.arange(1, len(eps3) + 1)
ax[1].loglog(k, eps3, '^-', color='C2', label=r'sublinear, $\epsilon_k = 1/k$')
ax[1].set_xlabel('k')
ax[1].set_ylabel(r"$\epsilon_k$")
ax[1].set_title("sublinear convergence (loglog)")
ax[1].legend()

plt.show()

* A `semilogy` plot (log scale on the $y$-axis only) shows $\log \epsilon_k$ against $k$.  Linear convergence $\epsilon_k \approx \mu^k \epsilon_0$ gives $\log \epsilon_k \approx k \log\mu + \log \epsilon_0$, which is a **straight line** with slope $\log\mu$.  Superlinear convergence **bends downward** (it gets steeper and steeper), and sublinear convergence **flattens out**.
* If a sublinear sequence behaves like $\epsilon_k \approx C k^{-p}$ for some power $p > 0$.  Then $\log \epsilon_k \approx \log C - p \log k$, which is a **straight line with slope $-p$ on a `loglog` plot**.  Here $\epsilon_k = 1/k$, so $p = 1$.

## Estimating the Order of Convergence

Now suppose we have a sequence of errors $\epsilon_k$ from an experiment, but we don't know the theory.  How can we measure $q$?

### Local Estimates

If $\epsilon_{k+1} \approx \mu\, \epsilon_k^q$, then taking logarithms gives

$$
\log \epsilon_{k+1} \approx q \log \epsilon_k + \log \mu
$$

Subtracting the same relation one step earlier ($\log \epsilon_k \approx q \log\epsilon_{k-1} + \log\mu$) eliminates the unknown $\mu$:

$$
\log\epsilon_{k+1} - \log\epsilon_k \approx q\,(\log\epsilon_k - \log\epsilon_{k-1})
\qquad\Longrightarrow\qquad
q \approx q_k = \frac{\log(\epsilon_{k+1}/\epsilon_k)}{\log(\epsilon_k/\epsilon_{k-1})}
$$

Each $q_k$ uses three consecutive errors, so we get one estimate per iteration, and we can watch the estimates settle down as $k$ grows.

In [ ]:
def local_q(eps):
    """
    local estimates q_k = log(eps[k+1]/eps[k]) / log(eps[k]/eps[k-1])
    for k = 1, ..., len(eps) - 2
    """
    r = np.diff(np.log(eps))  # r[k] = log(eps[k+1]) - log(eps[k])
    return r[1:] / r[:-1]

print("Linear:", np.round(local_q(eps1[:8]), 4))
print("Quadratic:", np.round(local_q(eps2[:8]), 4))
print("Sublinear:", np.round(local_q(eps3[:8]), 4))

### Fitting a Line

A more robust estimate uses all the data at once.  Let $r_k = \log\epsilon_{k+1} - \log\epsilon_k$.  The relation above says $r_k \approx q\, r_{k-1}$, so $|r_k| \approx |r_0|\, q^k$ grows (or shrinks) geometrically, and

$$
\log|r_k| \approx \log|r_0| + k \log q
$$

So if we plot $\log|r_k|$ against $k$, we should see a straight line with slope $\log q$.  The function below fits a line to these points using `np.polyfit`, and returns $q = e^{\text{slope}}$.

In [ ]:
def estimate_q(eps):
    """
    estimate rate of convergence q from sequence eps
    """
    x = np.arange(len(eps) -1)
    y = np.log(np.abs(np.diff(np.log(eps)))) # y[k] = log|r_k|
    line = np.polyfit(x, y, 1) # fit degree 1 polynomial
    q = np.exp(line[0]) # find q
    return q

In [ ]:
print(estimate_q(eps1)) # linear
print(estimate_q(eps2)) # quadratic
print(estimate_q(eps3)) # sublinear

For the model sequences, $\log \epsilon_{k+1} = q\log\epsilon_k + \log\mu$ holds exactly, so we recover $q = 1$ and $q = 2$ exactly.  For $\epsilon_k = 1/k$, the estimate is a bit below $1$; we'll come back to this in the exercises.

Even though the bisection errors jump around, fitting a line gives an estimate close to $1$.  For the secant method, dropping the first two (pre-asymptotic) errors moves the estimate closer to $\varphi \approx 1.618$.

We get the same orders ($1$, $1.618$, and $2$) and the same linear rate ($0.1716$) as before, without knowing the answer.

### Practical Tips

1. **Discard the pre-asymptotic phase.**  The order describes the behavior as $k \to \infty$.  The first few iterates may behave differently (e.g. the error of the secant method increased in the first step).
2. **Discard iterates at machine precision.**  In double precision, the error can't get much below $10^{-16}|x_\ast|$.  Once it gets there, round-off error dominates and the ratios are meaningless.
3. **Look at the plot first.**  A `semilogy` plot of $\epsilon_k$ (or $\delta_k$) against $k$ tells you a lot at a glance: a straight line means linear, bending down means superlinear, and flattening out means sublinear (in which case try a `loglog` plot).
4. **$q \approx 1$ is ambiguous.**  Check the rate $\mu$ to tell linear ($\mu < 1$) from sublinear ($\mu \to 1$) convergence.

### Exercises

1. Generate a converging sequence with $q = 1.5$, and estimate $q$ using the above methods.
2. Generate sublinear sequences with $\epsilon_k = 1/k$ (as above) for lengths 50, 100, 500, 1000, 5000, and 10000.  How does our estimate of $q$ change with the length of the sequence?

### Solutions

**Exercise 1.**

In [ ]:
# generates a sequence converging with order q = 1.5
n = 9
mu = 1.5
q = 1.5
eps4 = [0.2]
for i in range(n):
    eps4.append(mu * eps4[-1]**q)

plt.semilogy(eps4, 'o-')
plt.xlabel('k')
plt.ylabel(r"$\epsilon_k$")
plt.title("q = {} convergence".format(q))
plt.show()

print("estimate_q:", estimate_q(eps4))
print("local_q:   ", local_q(eps4))

**Exercise 2.**

In [ ]:
def gen_sublinear(n):
    eps = []
    for k in range(1,n):
        eps.append(1.0/k)
    return eps

ns = [50, 100, 500, 1000, 5000, 10000]
for n in ns:
    eps = gen_sublinear(n)
    print(n, estimate_q(eps))

The estimate approaches $1$ as the sequence gets longer, but slowly.  For $\epsilon_k = 1/k$, we have $r_k = \log\frac{k}{k+1} \approx -1/k$, so $\log|r_k| \approx -\log k$.  This is not a straight line in $k$, so fitting a line gives a slope slightly below $0$ (i.e. $q$ slightly below $1$).  The slope of $-\log k$ flattens out as $k$ grows, so the fitted slope gets closer to $0$ for longer sequences.  This is another reason to use a `loglog` plot for sublinear sequences.